In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import glob
from pathlib import Path
from array import array
import math
import ROOT

Welcome to JupyROOT 6.28/04


In [43]:
print("ok")

ok


In [2]:
from pathlib import Path

MIXTURE_CONFIGS = [
    {
        "name": "Ar_CO2_Iso_93_5_2",
        "base_path": Path("Ar_CO2_Iso_93_5_2"),
        "voltage_rules": {
            # regole specifiche opzionali per tensione
            # 545: {"disable_sig_floor": True},
            # 600: {"force_escape": True, "extend_right": True},
        },
    },
    {
        "name": "Ar_CO2_Iso_88_10_2",
        "base_path": Path("Ar_CO2_Iso_88_10_2"),
        "voltage_rules": {
            545: {
                "disable_sig_floor": True,
                "xmax_sigma_factor": 5.8,
            },
            600: {
                "force_escape": True,
                "extend_right": True,
                "extra_right_min": 120.0,
                "extra_right_sigma_factor": 1.5,
            },
        },
    },
]

print("Miscele configurate (ordine input/output):")
for cfg in MIXTURE_CONFIGS:
    print(f" - {cfg['name']}  ->  {cfg['base_path']}")

Miscele configurate (ordine input/output):
 - Ar_CO2_Iso_93_5_2  ->  Ar_CO2_Iso_93_5_2
 - Ar_CO2_Iso_88_10_2  ->  Ar_CO2_Iso_88_10_2


In [3]:
CHAMBERS = ["MM2DNAP1"]

def parse_txt_metadata(path_obj):
    name = path_obj.name
    voltage_match = re.search(r'_(\d+)V(?:_|\.txt)', name, flags=re.IGNORECASE)
    voltage = int(voltage_match.group(1)) if voltage_match else None

    chamber = next((c for c in CHAMBERS if c in name), None)
    is_bkg = "BKG" in name.upper()

    return {
        "path": path_obj,
        "name": name,
        "voltage": voltage,
        "chamber": chamber,
        "is_bkg": is_bkg,
    }


def build_pairs_for_mixture(base_path):
    all_txt_files = sorted(base_path.glob("Ar_Co2_Iso*.txt"))
    print(f"[{base_path.name}] trovati {len(all_txt_files)} file TXT Ar_Co2_Iso*.txt")

    pair_index = {ch: {} for ch in CHAMBERS}
    unknown_files = []

    for file_path in all_txt_files:
        meta = parse_txt_metadata(file_path)

        if meta["chamber"] is None or meta["voltage"] is None:
            unknown_files.append(meta["name"])
            continue

        chamber = meta["chamber"]
        voltage = meta["voltage"]

        if voltage not in pair_index[chamber]:
            pair_index[chamber][voltage] = {"bkg": None, "signal": None}

        slot = "bkg" if meta["is_bkg"] else "signal"
        existing = pair_index[chamber][voltage][slot]
        if existing is not None:
            print(f"[WARNING][{base_path.name}] duplicato {slot.upper()} per {chamber} {voltage}V: {existing.name} -> {meta['name']}")

        pair_index[chamber][voltage][slot] = meta["path"]

    paired_files = {ch: [] for ch in CHAMBERS}
    for chamber in CHAMBERS:
        for voltage in sorted(pair_index[chamber].keys()):
            item = pair_index[chamber][voltage]
            paired_files[chamber].append({
                "voltage": voltage,
                "signal_file": item["signal"],
                "bkg_file": item["bkg"],
                "complete_pair": bool(item["signal"] and item["bkg"]),
            })

    for chamber in CHAMBERS:
        complete = sum(1 for p in paired_files[chamber] if p["complete_pair"])
        total = len(paired_files[chamber])
        print(f"[{base_path.name}] {chamber}: {complete}/{total} coppie complete BKG/non-BKG")

    if unknown_files:
        print(f"\n[INFO][{base_path.name}] File ignorati (camera o tensione non riconosciute):")
        for name in unknown_files:
            print(" -", name)

    dataset = {
        chamber: [p for p in pairs if p["complete_pair"]]
        for chamber, pairs in paired_files.items()
    }

    return {
        "paired_files": paired_files,
        "dataset": dataset,
        "unknown_files": unknown_files,
    }


MIXTURE_DATA = {}
for cfg in MIXTURE_CONFIGS:
    mix_name = cfg["name"]
    base_path = cfg["base_path"]
    MIXTURE_DATA[mix_name] = build_pairs_for_mixture(base_path)

# compatibilità legacy (se qualche cella usa ancora questi nomi)
base_path = MIXTURE_CONFIGS[0]["base_path"]
paired_files = MIXTURE_DATA[MIXTURE_CONFIGS[0]["name"]]["paired_files"]
dataset = MIXTURE_DATA[MIXTURE_CONFIGS[0]["name"]]["dataset"]

[Ar_CO2_Iso_93_5_2] trovati 27 file TXT Ar_Co2_Iso*.txt
[Ar_CO2_Iso_93_5_2] MM2DNAP1: 11/16 coppie complete BKG/non-BKG
[Ar_CO2_Iso_88_10_2] trovati 25 file TXT Ar_Co2_Iso*.txt
[Ar_CO2_Iso_88_10_2] MM2DNAP1: 12/13 coppie complete BKG/non-BKG


In [4]:
import pandas as pd

all_reports = []
for cfg in MIXTURE_CONFIGS:
    mix_name = cfg["name"]
    base_path = cfg["base_path"]
    paired_files = MIXTURE_DATA[mix_name]["paired_files"]

    report_rows = []
    for chamber in CHAMBERS:
        for pair in paired_files[chamber]:
            signal_name = pair["signal_file"].name if pair["signal_file"] else "-"
            bkg_name = pair["bkg_file"].name if pair["bkg_file"] else "-"
            report_rows.append({
                "miscela": mix_name,
                "camera": chamber,
                "tensione_V": int(pair["voltage"]),
                "file_signal": signal_name,
                "file_BKG": bkg_name,
                "stato_pairing": "OK" if pair["complete_pair"] else "INCOMPLETO",
            })

    pairing_report_df = pd.DataFrame(report_rows).sort_values(
        ["miscela", "camera", "tensione_V"], ascending=[True, True, True]
    ).reset_index(drop=True)

    csv_report_path = base_path / "pairing_report.csv"
    pairing_report_df.to_csv(csv_report_path, index=False)
    all_reports.append(pairing_report_df)

    print(f"\n=== {mix_name} ===")
    print("Report pairing BKG/non-BKG:")
    display(pairing_report_df)
    print("\nRiepilogo per camera:")
    display(
        pairing_report_df.groupby(["camera", "stato_pairing"]).size().rename("n").reset_index()
    )
    print(f"CSV salvato in: {csv_report_path}")

pairing_report_all_df = pd.concat(all_reports, ignore_index=True) if all_reports else pd.DataFrame()
print("\n=== Riepilogo globale pairing ===")
if not pairing_report_all_df.empty:
    display(pairing_report_all_df.groupby(["miscela", "camera", "stato_pairing"]).size().rename("n").reset_index())
else:
    print("Nessun report disponibile.")


=== Ar_CO2_Iso_93_5_2 ===
Report pairing BKG/non-BKG:


,miscela,camera,tensione_V,file_signal,file_BKG,stato_pairing
0,Ar_CO2_Iso_93_5_2,MM2DNAP1,490,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
1,Ar_CO2_Iso_93_5_2,MM2DNAP1,500,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
2,Ar_CO2_Iso_93_5_2,MM2DNAP1,510,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
3,Ar_CO2_Iso_93_5_2,MM2DNAP1,515,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
4,Ar_CO2_Iso_93_5_2,MM2DNAP1,520,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
5,Ar_CO2_Iso_93_5_2,MM2DNAP1,525,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
6,Ar_CO2_Iso_93_5_2,MM2DNAP1,530,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
7,Ar_CO2_Iso_93_5_2,MM2DNAP1,535,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
8,Ar_CO2_Iso_93_5_2,MM2DNAP1,540,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK
9,Ar_CO2_Iso_93_5_2,MM2DNAP1,545,Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connes...,Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_connesso_Verd...,OK



Riepilogo per camera:


,camera,stato_pairing,n
0,MM2DNAP1,INCOMPLETO,5
1,MM2DNAP1,OK,11


CSV salvato in: Ar_CO2_Iso_93_5_2/pairing_report.csv

=== Ar_CO2_Iso_88_10_2 ===
Report pairing BKG/non-BKG:


,miscela,camera,tensione_V,file_signal,file_BKG,stato_pairing
0,Ar_CO2_Iso_88_10_2,MM2DNAP1,490,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
1,Ar_CO2_Iso_88_10_2,MM2DNAP1,500,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
2,Ar_CO2_Iso_88_10_2,MM2DNAP1,510,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
3,Ar_CO2_Iso_88_10_2,MM2DNAP1,520,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
4,Ar_CO2_Iso_88_10_2,MM2DNAP1,530,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
5,Ar_CO2_Iso_88_10_2,MM2DNAP1,540,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
6,Ar_CO2_Iso_88_10_2,MM2DNAP1,550,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
7,Ar_CO2_Iso_88_10_2,MM2DNAP1,560,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
8,Ar_CO2_Iso_88_10_2,MM2DNAP1,570,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...,OK
9,Ar_CO2_Iso_88_10_2,MM2DNAP1,575,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,-,INCOMPLETO



Riepilogo per camera:


,camera,stato_pairing,n
0,MM2DNAP1,INCOMPLETO,1
1,MM2DNAP1,OK,12


CSV salvato in: Ar_CO2_Iso_88_10_2/pairing_report.csv

=== Riepilogo globale pairing ===


,miscela,camera,stato_pairing,n
0,Ar_CO2_Iso_88_10_2,MM2DNAP1,INCOMPLETO,1
1,Ar_CO2_Iso_88_10_2,MM2DNAP1,OK,12
2,Ar_CO2_Iso_93_5_2,MM2DNAP1,INCOMPLETO,5
3,Ar_CO2_Iso_93_5_2,MM2DNAP1,OK,11


In [5]:
def subtract_background(signal_counts, bkg_counts):
    sig = np.asarray(signal_counts, dtype=float)
    bkg = np.asarray(bkg_counts, dtype=float)

    n = min(len(sig), len(bkg))
    if n == 0:
        return []

    corrected = sig[:n] - bkg[:n]
    corrected = np.clip(corrected, 0.0, None)
    return corrected.tolist()

In [48]:
# La conversione MCA -> TXT non serve più: ora lavoriamo direttamente sui file TXT.
print("Conversione MCA->TXT rimossa: utilizzo diretto di Ar_Co2_Iso*.txt")

Conversione MCA->TXT rimossa: utilizzo diretto di Ar_Co2_Iso*.txt


In [ ]:
for cfg in MIXTURE_CONFIGS:
    mix_name = cfg["name"]
    dataset_mix = MIXTURE_DATA[mix_name]["dataset"]
    print(f"\n=== {mix_name} ===")
    for chamber in CHAMBERS:
        print(f"{chamber}: {len(dataset_mix[chamber])} coppie complete pronte per il fit")

MM2DNAP1: 12 coppie complete pronte per il fit


In [50]:
def extract_voltage(fname):
    m = re.search(r'_(\d+)V\.txt$', os.path.basename(fname))
    return int(m.group(1)) if m else None

In [6]:
def read_counts_1col(fname):
    y = []
    data_started = False
    saw_data_markers = False

    with open(fname, "r", encoding="latin-1", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue

            su = s.upper()
            if "<<DATA>>" in su:
                data_started = True
                saw_data_markers = True
                continue
            if "<<END>>" in su:
                break
            if su.startswith("<<") and su.endswith(">>"):
                continue

            # Se il file contiene marker DATA/END, leggo solo dentro al blocco dati
            if saw_data_markers and not data_started:
                continue

            try:
                y.append(float(s))
            except ValueError:
                # ignora righe non numeriche (header/commenti)
                continue

    if not y:
        raise RuntimeError(f"Empty or unreadable spectrum data: {fname}")

    return y

In [7]:
def make_th1_from_counts(y, hname, title, trim_zeros=True):
    first = 0
    last = len(y) - 1

    if trim_zeros:
        while first < len(y) and y[first] == 0:
            first += 1
        while last >= 0 and y[last] == 0:
            last -= 1
        if last <= first:
            raise RuntimeError(f"After trimming zeros, no signal remains ({hname}).")

    nbins = last - first + 1
    h = ROOT.TH1F(hname, title, nbins, first - 0.5, last + 0.5)
    h.Sumw2()

    for i in range(nbins):
        ch = first + i
        c = float(y[ch])
        b = i + 1
        h.SetBinContent(b, c)
        # Poisson: per c=0 metto 0 => quei bin non pesano in chi2; con "L" è ok comunque
        h.SetBinError(b, math.sqrt(c) if c > 0 else 0.0)

    h.SetLineWidth(2)
    return h

In [10]:
def find_peak_base(h):

    peaks = find_1_or_2_peaks(h)

    if peaks:
        mu = max(peaks)
        return mu, "TSpectrum"

    # fallback semplice
    mu = h.GetBinCenter(h.GetMaximumBin())
    return mu, "MaxBin"


In [11]:
def check_peak_quality(h, mu):

    xmin = h.GetXaxis().GetXmin()
    xmax = h.GetXaxis().GetXmax()

    pos = (mu - xmin) / (xmax - xmin)

    # stima sigma
    fwhm, _, _ = estimate_fwhm_from_hist(h, mu)
    sigma = max(5.0, fwhm / 2.355)

    b = h.FindBin(mu)
    y_peak = h.GetBinContent(b)

    # ✅ scala con sigma (non numeri fissi tipo 20)
    shift = int(max(5, sigma))

    left = h.GetBinContent(max(1, b - shift))
    right = h.GetBinContent(min(h.GetNbinsX(), b + shift))

    background = 0.5 * (left + right)

    # ✅ definizione migliore di SNR
    snr = (y_peak - background) / max(1.0, background)

    # ----------------------------------
    # HARD cuts (pochissimi!)
    # ----------------------------------

    if pos < 0.03:
        return "BAD", {"reason": "bordo", "snr": snr}

    if sigma < 5:
        return "BAD", {"reason": "spike", "snr": snr}

    # ----------------------------------
    # SNR → SOLO informativo
    # ----------------------------------

    if snr < 0.3:
        return "MEDIUM", {"reason": "snr basso", "snr": snr, "sigma": sigma}

    return "GOOD", {"snr": snr, "sigma": sigma}

In [12]:
def find_peak_after_valley(h):

    import numpy as np

    nb = h.GetNbinsX()

    # smoothing leggero
    h_s = h.Clone()
    h_s.Smooth(5)

    y = np.array([h_s.GetBinContent(i+1) for i in range(nb)])
    dy = np.diff(y)

    # -------------------------
    # cerca valle (minimo locale)
    # -------------------------
    valley_bin = None

    for i in range(5, nb-5):

        if dy[i-1] < 0 and dy[i] > 0:
            valley_bin = i + 1
            break

    if valley_bin is None:
        print("[FALLBACK] no valley → max globale")
        b = h.GetMaximumBin()
        return h.GetBinCenter(b)

    x_valley = h.GetBinCenter(valley_bin)
    print(f"[VALLEY] x = {x_valley:.1f}")

    # -------------------------
    # massimo DOPO valle
    # -------------------------
    best_bin = valley_bin
    best_val = 0

    for b in range(valley_bin, nb+1):

        val = h.GetBinContent(b)

        if val > best_val:
            best_val = val
            best_bin = b

    mu = h.GetBinCenter(best_bin)

    print(f"[FALLBACK] mu = {mu:.1f}")

    return mu

In [13]:
def find_xmin_first_valley(h, mu):

    h_s = h.Clone()
    h_s.Smooth(5)

    nb = h_s.GetNbinsX()

    # ---------------------
    # spike iniziale
    # ---------------------
    spike_bin = 1
    spike_val = 0

    for b in range(1, int(0.4 * nb)):
        y = h_s.GetBinContent(b)
        if y > spike_val:
            spike_val = y
            spike_bin = b

    # ---------------------
    # primo minimo dopo spike
    # ---------------------
    valley_bin = spike_bin
    last = h_s.GetBinContent(spike_bin)

    for b in range(spike_bin + 1, nb):

        y = h_s.GetBinContent(b)

        if y < last:
            valley_bin = b
            last = y
        else:
            break

    xmin = h.GetBinCenter(valley_bin)

    print(f"[XMIN] valley → {xmin:.1f}")

    return xmin

In [8]:
def estimate_fwhm_from_hist(h, mu, window_bins=1200, smooth_for_width=3):
    """
    Stima FWHM attorno a mu.
    Restituisce (fwhm, x_peak, y_peak).
    """
    hw = h
    if smooth_for_width and smooth_for_width > 0:
        hw = h.Clone(h.GetName() + "_width")
        hw.Smooth(smooth_for_width)

    b0 = hw.FindBin(mu)
    nb = hw.GetNbinsX()
    bmin = max(1, b0 - window_bins)
    bmax = min(nb, b0 + window_bins)

    # massimo locale nel window
    bpeak = b0
    ypeak = -1.0
    for b in range(bmin, bmax + 1):
        yb = hw.GetBinContent(b)
        if yb > ypeak:
            ypeak = yb
            bpeak = b

    if ypeak <= 0:
        return (0.0, hw.GetBinCenter(b0), 0.0)

    xpeak = hw.GetBinCenter(bpeak)
    half = 0.5 * ypeak

    # trova crossing sinistro
    bl = bpeak
    while bl > bmin and hw.GetBinContent(bl) > half:
        bl -= 1

    # trova crossing destro
    br = bpeak
    while br < bmax and hw.GetBinContent(br) > half:
        br += 1

    # se non trovi crossing, fallback
    if bl == bmin or br == bmax:
        # ritorna una scala “larga” ma non folle
        width = hw.GetBinCenter(bmax) - hw.GetBinCenter(bmin)
        return (max(1.0, width/5.0), xpeak, ypeak)

    def x_at_half(b_low, b_high):
        x1 = hw.GetBinCenter(b_low);  y1 = hw.GetBinContent(b_low)
        x2 = hw.GetBinCenter(b_high); y2 = hw.GetBinContent(b_high)
        if y2 == y1:
            return x1
        t = (half - y1) / (y2 - y1)
        return x1 + t*(x2 - x1)

    xL = x_at_half(bl, bl + 1)
    xR = x_at_half(br - 1, br)
    fwhm = max(1e-6, xR - xL)
    return (fwhm, xpeak, ypeak)


In [58]:
def fwhm_to_sigma(fwhm):
    return float(fwhm) / 2.355

In [9]:
# ------------------------------------------------------------
# Peak finding: TSpectrum su copia rebinnata/smussata
#    -> restituisce 1 o 2 picchi (canali interi), ordinati per altezza decrescente
# ------------------------------------------------------------
def find_1_or_2_peaks(h, maxpeaks=20, threshold=0.05, sigma_find=6.0,
                     rebin_target=10, smooth_find=5,
                     require_separation=True):
    """
    Ritorna:
      peaks = [muA] oppure [muA, muB] (canali interi), in ordine di altezza (muA = più alto).
    """
    # copia per TSpectrum
    h_find = h.Clone(h.GetName() + "_find")

    # rebin: scelgo un divisore vicino a rebin_target (semplice e stabile)
    nb = h_find.GetNbinsX()
    r = int(rebin_target)
    while r > 1 and (nb % r != 0):
        r -= 1
    if r > 1:
        h_find.Rebin(r)

    if smooth_find and smooth_find > 0:
        h_find.Smooth(smooth_find)

    # sigma_find non deve essere enorme rispetto a nbins rebinnati
    nb2 = h_find.GetNbinsX()
    #sigma_use = min(float(sigma_find), max(2.0, nb2/15.0))
    sigma_use = max(2.0, min(float(sigma_find), nb2/15.0))
    
    spec = ROOT.TSpectrum(maxpeaks)
    nfound = spec.Search(h_find, sigma_use, "nodraw", float(threshold))

    if nfound <= 0:
        return []

    xs = [float(spec.GetPositionX()[i]) for i in range(nfound)]

    # converti a canali interi sullo spettro originale e calcola “altezza” reale
    cand = []
    for x in xs:
        b = h.FindBin(x)
        mu = int(round(h.GetBinCenter(b)))
        y  = float(h.GetBinContent(b))
        cand.append((mu, y))

    # unique per mu tenendo la y max
    best = {}
    for mu, y in cand:
        if (mu not in best) or (y > best[mu]):
            best[mu] = y

    cand2 = sorted([(mu, y) for mu, y in best.items()], key=lambda t: t[1], reverse=True)
    '''
    # filtra picchi troppo a sinistra (fondo)
    cut_min = int(0.35 * h.GetNbinsX())   # ~15% asse
    cand2 = [(mu, y) for (mu, y) in cand2 if mu > cut_min]
    '''
    if not cand2:
        return []

    # prendo il più alto
    muA, yA = cand2[0]
    peaks = [muA]

    # provo a trovare il secondo: se require_separation, impongo separazione > ~0.8*FWHM(main)
    if len(cand2) >= 2:
        # stima FWHM del principale per imporre distanza minima (evita dentini)
        fwhmA, _, _ = estimate_fwhm_from_hist(h, muA, window_bins=1200, smooth_for_width=3)
        min_sep = 0.8 * fwhmA if require_separation else 0.0
        if min_sep < 50:   # guardrail minimo
            min_sep = 50.0

        muB = None
        for mu, y in cand2[1:]:
            if abs(mu - muA) >= min_sep:
                muB = mu
                break

        if muB is not None:
            peaks.append(muB)

    return peaks


In [14]:
def draw_components(h, f_tot, canvas, use_escape):

    h.Draw("HIST")

    if use_escape:
        # -------------------------
        # componenti doppie
        # -------------------------
        f_main = f_tot.Clone("f_main")
        f_escape = f_tot.Clone("f_escape")
        f_bkg = f_tot.Clone("f_bkg")

        # spegni componenti
        f_main.SetParameter(3, 0)
        f_main.SetParameter(6, 0)

        f_escape.SetParameter(0, 0)
        f_escape.SetParameter(6, 0)

        f_bkg.SetParameter(0, 0)
        f_bkg.SetParameter(3, 0)

        # stili
        f_main.SetLineColor(ROOT.kRed)
        f_main.SetLineWidth(3)

        f_escape.SetLineColor(ROOT.kOrange+7)
        f_escape.SetLineStyle(7)

        f_bkg.SetLineColor(ROOT.kGreen+2)
        f_bkg.SetLineWidth(3)

        f_bkg.Draw("SAME")
        f_escape.Draw("SAME")
        f_main.Draw("SAME")

        canvas._keep = [f_main, f_escape, f_bkg]

    else:
        # -------------------------
        # solo main + bkg
        # -------------------------
        f_main = f_tot.Clone("f_main")
        f_bkg = f_tot.Clone("f_bkg")

        f_main.SetParameter(3, 0)
        f_bkg.SetParameter(0, 0)

        f_main.SetLineColor(ROOT.kRed)
        f_main.SetLineWidth(3)

        f_bkg.SetLineColor(ROOT.kGreen+2)
        f_bkg.SetLineWidth(3)

        f_bkg.Draw("SAME")
        f_main.Draw("SAME")

        canvas._keep = [f_main, f_bkg]

    # totale SEMPRE visibile
    f_tot.SetLineColor(ROOT.kYellow)
    f_tot.SetLineWidth(4)
    f_tot.Draw("SAME")

In [15]:
def draw_text(f_tot, use_escape):

    latex = ROOT.TLatex()
    latex.SetNDC(True)
    latex.SetTextSize(0.03)

    x = 0.62
    y = 0.85
    dy = 0.05

    # main
    latex.DrawLatex(x, y,
        f"#mu_main = {f_tot.GetParameter(1):.1f}")
    y -= dy

    latex.DrawLatex(x, y,
        f"#sigma_main = {f_tot.GetParameter(2):.1f}")
    y -= dy

    # escape SOLO se usato
    if use_escape:
        latex.DrawLatex(x, y,
            f"#mu_escape = {f_tot.GetParameter(4):.1f}")
        y -= dy

        latex.DrawLatex(x, y,
            f"#sigma_escape = {f_tot.GetParameter(5):.1f}")
        y -= dy

    # chi2
    chi2 = f_tot.GetChisquare()
    ndf  = f_tot.GetNDF()

    latex.DrawLatex(x, y,
        f"#chi^{{2}}/ndf = {chi2/ndf:.2f}")

In [16]:
def fit_best_model(h, mu, xmin, xmax, force_escape=False):

    import ROOT

    # ----------------------------------
    # ✅ sigma iniziale ROBUSTA
    # ----------------------------------
    fwhm, _, _ = estimate_fwhm_from_hist(h, mu)
    sigma0 = max(5.0, fwhm / 2.355)
    sigma_escape0 = max(12.0, 0.8 * sigma0)

    print(f"[DEBUG] mu={mu:.1f} sigma0={sigma0:.1f} sigma_escape0={sigma_escape0:.1f} xmin={xmin:.1f} xmax={xmax:.1f}")

    A0 = h.GetBinContent(h.FindBin(mu))

    # diagnostica segnale debole: evita fit instabili su picchi molto bassi
    bmu = h.FindBin(mu)
    delta_b = max(3, int(max(5.0, sigma0)))
    yL_mu = h.GetBinContent(max(1, bmu - delta_b))
    yR_mu = h.GetBinContent(min(h.GetNbinsX(), bmu + delta_b))
    y_bg_mu = max(1.0, 0.5 * (yL_mu + yR_mu))
    snr_mu = (A0 - y_bg_mu) / y_bg_mu
    weak_signal = (snr_mu < 0.40)
    print(f"[WEAK] snr_mu={snr_mu:.2f} weak={weak_signal}")

    # limiti più fisici su sigma (utile nei segnali deboli)
    sigma_min = max(0.5 * sigma0, 5.0)
    sigma_max = min(2.2 * sigma0, 0.30 * max(mu, 1.0))
    if weak_signal:
        sigma_max = min(sigma_max, 0.22 * max(mu, 1.0))
    if sigma_max <= sigma_min:
        sigma_max = sigma_min + 2.0

    # ----------------------------------
    # stima iniziale fondo dai bordi del fit range
    # ----------------------------------
    bL = h.FindBin(xmin)
    bR = h.FindBin(xmax)
    yL = max(1.0, h.GetBinContent(bL))
    yR = max(1.0, h.GetBinContent(bR))

    dx = max(1.0, xmax - xmin)
    bg_exp0 = math.log(yR / yL) / dx
    bg_exp0 = max(-0.02, min(0.02, bg_exp0))
    bg_amp0 = max(1.0, min(yL, yR) * 0.8)

    # ===============================
    # ✅ FIT 1 → UNA GAUSS
    # ===============================
    z_expr = f"(2*x - {xmin:.6f} - {xmax:.6f})/({xmax:.6f} - {xmin:.6f})"
    bg1 = f"[3]*exp([4]*x) * (1 + [5]*{z_expr} + [6]*(2*{z_expr}*{z_expr} - 1))"

    f1 = ROOT.TF1(
        f"f1_{h.GetName()}",
        "[0]*exp(-0.5*((x-[1])/[2])^2) + " + bg1,
        xmin, xmax
    )

    # parameters: A_main, mu, sigma_main, bg_amp, bg_exp, cheb1, cheb2
    f1.SetParameters(A0, mu, sigma0, bg_amp0, bg_exp0, 0.0, 0.0)

    f1.SetParLimits(2, sigma_min, sigma_max)
    f1.SetParLimits(3, 0.0, 5.0 * max(1.0, A0))
    f1.SetParLimits(4, -0.02, 0.02)
    f1.SetParLimits(5, -0.4, 0.4)
    f1.SetParLimits(6, -0.4, 0.4)

    # 🔥 FIT USANDO SOLO IL RANGE
    h.Fit(f1, "RQ0", "", xmin, xmax)

    chi2_1 = f1.GetChisquare()
    ndf_1  = f1.GetNDF()
    red1 = chi2_1/ndf_1 if ndf_1 > 0 else 999

    # ===============================
    # ✅ FIT 2 → DUE GAUSS
    # ===============================
    if force_escape:
        # solo 600V: posizione fisica attesa dell'escape più a sinistra
        mu_escape = 0.52 * mu
        mu_escape_low = 0.46 * mu
        mu_escape_high = 0.58 * mu
    else:
        mu_escape = 0.65 * mu
        mu_escape_low = 0.58 * mu
        mu_escape_high = 0.70 * mu

    f2 = ROOT.TF1(
        f"f2_{h.GetName()}",
        "[0]*exp(-0.5*((x-[1])/[2])^2) +"
        "[3]*exp(-0.5*((x-[4])/[5])^2) +"
        "[6]*exp([7]*x) * (1 + [8]*{z} + [9]*(2*{z}*{z} - 1))".format(z=z_expr),
        xmin, xmax
    )

    # parameters: A_main, mu, sigma_main, A_escape, mu_escape, sigma_escape, bg_amp, bg_exp, cheb1, cheb2
    f2.SetParameters(
        A0, mu, sigma0,
        0.20*A0, mu_escape, sigma_escape0,
        bg_amp0, bg_exp0, 0.0, 0.0
    )

    f2.SetParLimits(2, sigma_min, sigma_max)
    if force_escape:
        # Solo 600V: fondo più vincolato + escape non troppo largo
        f2.SetParLimits(5, 0.6 * sigma0, 3.0 * sigma0)
        f2.SetParLimits(3, 0.0, 0.55*A0)
        f2.SetParLimits(6, 0.0, 1.8 * max(1.0, bg_amp0))
        f2.SetParLimits(7, -0.03, 0.0)
        f2.SetParLimits(8, -0.10, 0.0)
        f2.SetParLimits(9, -0.10, 0.0)
    else:
        f2.SetParLimits(5, 0.6*sigma_min, 1.8*sigma_max)
        f2.SetParLimits(3, 0.0, 0.35*A0)
        f2.SetParLimits(7, -0.02, 0.02)
        f2.SetParLimits(8, -0.4, 0.4)
        f2.SetParLimits(9, -0.4, 0.4)
        f2.SetParLimits(6, 0.0, 5.0 * max(1.0, A0))

    f2.SetParLimits(4, mu_escape_low, mu_escape_high)

    # 🔥 FIT NEL RANGE GIUSTO
    h.Fit(f2, "RQ0", "", xmin, xmax)

    chi2_2 = f2.GetChisquare()
    ndf_2  = f2.GetNDF()
    red2 = chi2_2/ndf_2 if ndf_2 > 0 else 999

    # se il segnale è debole, forziamo modello semplice (main + bkg)
    if weak_signal and (not force_escape):
        print(f"[MODEL] weak-signal -> 1-gauss chi2/ndf={red1:.2f}")
        return f1, xmin, False

    amp_escape = abs(f2.GetParameter(3))
    mu_escape_fit = f2.GetParameter(4)
    sigma_escape_fit = f2.GetParameter(5)

    # visibilità reale dell'escape sopra il fondo locale
    f2_bkg = f2.Clone(f"f2bkg_{h.GetName()}")
    f2_bkg.SetParameter(0, 0.0)
    f2_bkg.SetParameter(3, 0.0)
    y_bkg_at_escape = max(1e-6, f2_bkg.Eval(mu_escape_fit))
    y_bkg_at_main = max(1e-6, f2_bkg.Eval(mu))
    escape_visibility = amp_escape / y_bkg_at_escape
    bkg_main_frac = y_bkg_at_main / max(1e-6, A0)

    aic1 = chi2_1 + 2.0 * 5.0
    aic2 = chi2_2 + 2.0 * 8.0

    good_escape = (
        (0.58*mu < mu_escape_fit < 0.70*mu) and
        (amp_escape > 0.12 * A0) and
        (0.5 * sigma0 < sigma_escape_fit < 1.6 * sigma0) and
        (red2 < 0.95 * red1) and
        (escape_visibility > 0.9)
    )

    # Modalità più permissiva SOLO quando richiesto (usata nel notebook solo per 600V)
    if force_escape and (not good_escape):
        good_escape = (
            (0.46*mu < mu_escape_fit < 0.58*mu) and
            (amp_escape > 0.05 * A0) and
            (0.6 * sigma0 < sigma_escape_fit < 3.0 * sigma0) and
            (red2 < 1.12 * red1) and
            (bkg_main_frac < 0.42)
        )

    print(
        f"[COMPARE] 1g red={red1:.2f} AIC={aic1:.1f}  2g red={red2:.2f} AIC={aic2:.1f} "
        f"amp_escape={amp_escape:.1f} mu_escape={mu_escape_fit:.1f} sigma_escape={sigma_escape_fit:.1f} "
        f"vis_escape={escape_visibility:.2f} bkg@main/A0={bkg_main_frac:.2f} force_escape={force_escape}"
    )

    # Se force_escape=True, accettiamo 2-gauss quando fisicamente plausibile e con perdita di fit minima
    if force_escape:
        physical_escape_600 = (
            (0.46*mu < mu_escape_fit < 0.58*mu) and
            (amp_escape > 0.05 * A0) and
            (0.6 * sigma0 < sigma_escape_fit < 3.2 * sigma0) and
            (bkg_main_frac < 0.45)
        )
        choose_2g = physical_escape_600 and (red2 <= 1.03 * red1)
    else:
        choose_2g = (aic2 < aic1 - 4.0) and good_escape

    if choose_2g:
        f_tot = f2
        use_escape = True
        print(f"[MODEL] 2-gauss chi2/ndf={chi2_2/ndf_2:.2f}")
    else:
        f_tot = f1
        use_escape = False
        print(f"[MODEL] 1-gauss chi2/ndf={chi2_1/ndf_1:.2f}")

    # ===============================
    # ✅ controllo stabilità
    # ===============================
    mu_fit = f_tot.GetParameter(1)
    sigma_fit = f_tot.GetParameter(2)

    if (mu_fit < xmin) or (mu_fit > xmax) or (sigma_fit <= 0):
        print("[FIT] instabile → rifit largo")

        xmax2 = h.GetXaxis().GetXmax()
        h.Fit(f_tot, "RQ0", "", xmin, xmax2)

    return f_tot, xmin, use_escape

In [17]:
def compute_residuals(h, f_tot, xmin, xmax):
    """
    Calcola i residui normalizzati (pull) SOLO nel fit range.
    Restituisce (residuals_hist, residuals_list).
    """
    import ROOT

    nbins = h.GetNbinsX()
    bmin = max(1, h.FindBin(xmin))
    bmax = min(nbins, h.FindBin(xmax))

    if bmax < bmin:
        bmid = max(1, min(nbins, h.FindBin(0.5 * (xmin + xmax))))
        bmin = bmid
        bmax = bmid

    residuals = []

    xlo = h.GetBinLowEdge(bmin)
    xhi = h.GetBinLowEdge(bmax + 1)
    nb_fit = max(1, bmax - bmin + 1)

    h_residuals = ROOT.TH1F(
        f"h_residuals_{h.GetName()}",
        f"Residuals (Pull) - {h.GetTitle()};Canale;(data-fit)/#sigma",
        nb_fit,
        xlo,
        xhi
    )

    for b in range(bmin, bmax + 1):
        x = h.GetBinCenter(b)
        y_data = h.GetBinContent(b)
        y_error = h.GetBinError(b)
        y_fit = f_tot.Eval(x)

        if y_error > 0:
            pull = (y_data - y_fit) / y_error
        else:
            pull = 0.0 if y_data == y_fit else (y_data - y_fit)

        ib = b - bmin + 1
        h_residuals.SetBinContent(ib, pull)
        h_residuals.SetBinError(ib, 1.0)
        residuals.append(pull)

    h_residuals.SetLineColor(ROOT.kBlue)
    h_residuals.SetLineWidth(2)
    h_residuals.SetMarkerStyle(20)
    h_residuals.SetMarkerSize(0.6)

    return h_residuals, residuals


In [20]:
def main():

    fit_summary_rows = []

    for mix_cfg in MIXTURE_CONFIGS:

        gas_mixture = mix_cfg["name"]
        voltage_rules = mix_cfg.get("voltage_rules", {})

        print("\n############################################")
        print(f"Processing mixture: {gas_mixture}")
        print("############################################")

        dataset_mix = MIXTURE_DATA.get(gas_mixture, {}).get("dataset", {})

        for chamber in ["MM2DNAP1"]:

            print("\n==============================")
            print(f"Processing chamber: {chamber}")
            print("==============================")

            pairs = dataset_mix.get(chamber, [])

            if not pairs:
                print(f"[WARNING] Nessuna coppia completa BKG/non-BKG per {gas_mixture} {chamber}")
                continue

            outname = f"output_{gas_mixture}_{chamber}.root"
            fout = ROOT.TFile(outname, "RECREATE")

            # cartelle output PNG
            png_root = Path("output") / gas_mixture / "fit"
            fit_results_dir = png_root / "fit_results"
            bkg_histo_dir = png_root / "bkg_histo"
            tot_histo_dir = png_root / "tot_histo"

            fit_results_dir.mkdir(parents=True, exist_ok=True)
            bkg_histo_dir.mkdir(parents=True, exist_ok=True)
            tot_histo_dir.mkdir(parents=True, exist_ok=True)

            print(f"[PNG] fit_results -> {fit_results_dir}")
            print(f"[PNG] bkg_histo  -> {bkg_histo_dir}")
            print(f"[PNG] tot_histo  -> {tot_histo_dir}")

            t = ROOT.TTree("fitResults", f"Fe55 fit results - {gas_mixture} - {chamber}")

            v_V = array("i", [0])
            v_mu1 = array("f", [0.0])
            v_s1 = array("f", [0.0])
            v_A1 = array("f", [0.0])
            v_B = array("f", [0.0])
            v_tau = array("f", [0.0])
            v_chi2 = array("f", [0.0])
            v_ndf = array("i", [0])
            v_c2ndf = array("f", [0.0])

            t.Branch("V", v_V, "V/I")
            t.Branch("mu1", v_mu1, "mu1/F")
            t.Branch("sigma1", v_s1, "sigma1/F")
            t.Branch("A1", v_A1, "A1/F")
            t.Branch("B", v_B, "B/F")
            t.Branch("tau", v_tau, "tau/F")
            t.Branch("chi2", v_chi2, "chi2/F")
            t.Branch("ndf", v_ndf, "ndf/I")
            t.Branch("chi2ndf", v_c2ndf, "chi2ndf/F")

            # --------------------------------------------------
            # LOOP COPPIE BKG/non-BKG
            # --------------------------------------------------
            for pair in pairs:

                V = int(pair["voltage"])
                signal_file = pair["signal_file"]
                bkg_file = pair["bkg_file"]
                vrule = voltage_rules.get(V, {})

                if signal_file is None or bkg_file is None:
                    print(f"[SKIP] Pair incompleta per {gas_mixture} {chamber} {V}V")
                    continue

                y_signal = read_counts_1col(signal_file)
                y_bkg = read_counts_1col(bkg_file)

                h_sig_raw_name = f"h_sigraw_{gas_mixture}_{chamber}_{V}V"
                h_sig_raw_title = f"{gas_mixture} {chamber} {V}V Signal raw;Canale;Conteggi"
                h_sig_raw = make_th1_from_counts(y_signal, h_sig_raw_name, h_sig_raw_title, trim_zeros=False)

                h_bkg_name = f"h_bkg_{gas_mixture}_{chamber}_{V}V"
                h_bkg_title = f"{gas_mixture} {chamber} {V}V BKG;Canale;Conteggi"
                h_bkg = make_th1_from_counts(y_bkg, h_bkg_name, h_bkg_title, trim_zeros=False)

                hname = f"h_{gas_mixture}_{chamber}_{V}V"
                title = f"{gas_mixture} {chamber} {V}V (signal - BKG);Canale;Conteggi"
                h = h_sig_raw.Clone(hname)
                h.SetTitle(title)
                h.Add(h_bkg, -1.0)
                for b in range(1, h.GetNbinsX() + 1):
                    if h.GetBinContent(b) < 0.0:
                        h.SetBinContent(b, 0.0)
                        h.SetBinError(b, 0.0)
                h.SetLineWidth(2)

                if h.GetMaximum() <= 0:
                    print(f"[SKIP] Spettro nullo dopo sottrazione BKG: {gas_mixture} {chamber} {V}V")
                    continue

                # --------------------------------------------------
                # PEAK FINDING ROBUSTO
                # --------------------------------------------------
                mu, method = find_peak_base(h)
                print(f"[BASE] {gas_mixture} {chamber} {V}V mu={mu:.1f} ({method})")

                quality, info = check_peak_quality(h, mu)
                print(f"[QUALITY] {quality} {info}")

                if quality == "BAD":
                    print("[FALLBACK]")
                    mu = find_peak_after_valley(h)

                # --- sigma ---
                fwhm, _, _ = estimate_fwhm_from_hist(h, mu)
                sigma0 = max(5.0, fwhm / 2.355)

                # --- xmin ---
                xmin = find_xmin_first_valley(h, mu)
                xmin = min(xmin, mu - 1.0 * sigma0)
                xmin = max(xmin, 0.1 * mu)

                # vincolo: xmin su bin significativo SOLO se il picco è molto largo (casi tipo 490V)
                # oppure disattivato da regola specifica (es. 545V)
                disable_sig_floor = bool(vrule.get("disable_sig_floor", False))
                use_sig_floor = ((sigma0 / max(1.0, mu)) > 0.25) and (not disable_sig_floor)
                if use_sig_floor:
                    b_mu = h.FindBin(mu)
                    y_peak = h.GetBinContent(b_mu)
                    thr_sig = max(1.0, 0.08 * y_peak)
                    b_sig_left = None
                    for b in range(1, b_mu + 1):
                        if h.GetBinContent(b) >= thr_sig:
                            b_sig_left = b
                            break
                    if b_sig_left is not None:
                        xmin_sig = h.GetBinCenter(b_sig_left)
                        xmin = max(xmin, xmin_sig)
                        print(f"[XMIN_SIG] threshold={thr_sig:.1f} -> xmin>={xmin_sig:.1f}")
                else:
                    reason = "regola specifica" if disable_sig_floor else "weak-signal / picco non largo"
                    print(f"[XMIN_SIG] skip ({reason}): tengo xmin meno aggressivo")

                # --- xmax ---
                xmax_sigma_factor = float(vrule.get("xmax_sigma_factor", 5.0))
                xmax = mu + xmax_sigma_factor * sigma0

                extend_right = bool(vrule.get("extend_right", V == 600))
                if extend_right:
                    extra_right_min = float(vrule.get("extra_right_min", 120.0))
                    extra_right_sigma_factor = float(vrule.get("extra_right_sigma_factor", 1.5))
                    extra_right = max(extra_right_min, extra_right_sigma_factor * sigma0)
                    xmax = min(h.GetXaxis().GetXmax(), xmax + extra_right)
                    print(f"[RANGE {V}V] estendo margine destro: +{extra_right:.1f} -> xmax={xmax:.1f}")

                if xmin >= xmax:
                    print("[FIX] range invalido")
                    xmin = mu - 2.0 * sigma0
                    xmax = mu + xmax_sigma_factor * sigma0
                    if extend_right:
                        extra_right_min = float(vrule.get("extra_right_min", 120.0))
                        extra_right_sigma_factor = float(vrule.get("extra_right_sigma_factor", 1.5))
                        extra_right = max(extra_right_min, extra_right_sigma_factor * sigma0)
                        xmax = min(h.GetXaxis().GetXmax(), xmax + extra_right)
                        print(f"[RANGE {V}V] estendo margine destro (fix-range): +{extra_right:.1f} -> xmax={xmax:.1f}")

                force_escape_this_voltage = bool(vrule.get("force_escape", V == 600))
                f_tot, xmin, use_escape = fit_best_model(h, mu, xmin, xmax, force_escape=force_escape_this_voltage)

                # --------------------------------------------------
                # RESIDUI
                # --------------------------------------------------
                h_residuals, residuals_list = compute_residuals(h, f_tot, xmin, xmax)

                # --------------------------------------------------
                # CANVAS
                # --------------------------------------------------
                c = ROOT.TCanvas(f"c_{gas_mixture}_{chamber}_{V}V", f"{gas_mixture} {chamber} {V}V", 1000, 650)
                c.SetGrid()

                draw_components(h, f_tot, c, use_escape)

                line = ROOT.TLine(xmin, 0, xmin, h.GetMaximum())
                line.SetLineColor(ROOT.kBlack)
                line.SetLineStyle(2)
                line.SetLineWidth(2)
                line.Draw("SAME")

                draw_text(f_tot, use_escape)

                # --------------------------------------------------
                # CANVAS RESIDUI
                # --------------------------------------------------
                c_res = ROOT.TCanvas(f"c_residuals_{gas_mixture}_{chamber}_{V}V", f"Residuals - {gas_mixture} {chamber} {V}V", 1000, 500)
                h_residuals.Draw("P")

                # linea di riferimento a 0
                line_res = ROOT.TLine(h_residuals.GetXaxis().GetXmin(), 0, h_residuals.GetXaxis().GetXmax(), 0)
                line_res.SetLineColor(ROOT.kRed)
                line_res.SetLineStyle(2)
                line_res.SetLineWidth(2)
                line_res.Draw("SAME")

                # --------------------------------------------------
                # SALVATAGGIO ROOT
                # --------------------------------------------------
                h_bkg.Write()
                h.Write()
                h_residuals.Write()
                f_tot.Write()
                c.Write()
                c_res.Write()

                # --------------------------------------------------
                # EXPORT PNG
                # --------------------------------------------------
                fit_png = fit_results_dir / f"fit_{chamber}_{V}V.png"
                res_png = fit_results_dir / f"residuals_{chamber}_{V}V.png"
                c.SaveAs(str(fit_png))
                c_res.SaveAs(str(res_png))

                c_bkg = ROOT.TCanvas(f"c_png_bkg_{gas_mixture}_{chamber}_{V}V", f"BKG {gas_mixture} {chamber} {V}V", 1000, 650)
                c_bkg.SetGrid()
                h_bkg.Draw("HIST")
                bkg_png = bkg_histo_dir / f"bkg_{chamber}_{V}V.png"
                c_bkg.SaveAs(str(bkg_png))

                c_tot = ROOT.TCanvas(f"c_png_tot_{gas_mixture}_{chamber}_{V}V", f"TOT {gas_mixture} {chamber} {V}V", 1000, 650)
                c_tot.SetGrid()
                h.Draw("HIST")
                tot_png = tot_histo_dir / f"tot_{chamber}_{V}V.png"
                c_tot.SaveAs(str(tot_png))

                c_bkg.Close()
                c_tot.Close()

                # --------------------------------------------------
                # TREE
                # --------------------------------------------------
                v_V[0] = int(V)
                v_mu1[0] = float(f_tot.GetParameter(1))
                v_s1[0]  = float(f_tot.GetParameter(2))
                v_A1[0]  = float(f_tot.GetParameter(0))

                if use_escape:
                    v_B[0]   = float(f_tot.GetParameter(6))
                    v_tau[0] = float(f_tot.GetParameter(7))
                    model_used = "2gauss_escape"
                else:
                    v_B[0]   = float(f_tot.GetParameter(3))
                    v_tau[0] = float(f_tot.GetParameter(4))
                    model_used = "1gauss"

                chi2 = f_tot.GetChisquare()
                ndf = f_tot.GetNDF()
                chi2ndf = float(chi2 / ndf) if ndf > 0 else 0.0

                v_chi2[0] = float(chi2)
                v_ndf[0] = int(ndf)
                v_c2ndf[0] = chi2ndf

                t.Fill()

                fit_summary_rows.append({
                    "miscela": gas_mixture,
                    "camera": chamber,
                    "tensione_V": int(V),
                    "mu": float(v_mu1[0]),
                    "sigma": float(v_s1[0]),
                    "chi2": float(v_chi2[0]),
                    "ndf": int(v_ndf[0]),
                    "chi2_ndf": float(v_c2ndf[0]),
                    "modello": model_used,
                    "signal_file": signal_file.name,
                    "bkg_file": bkg_file.name,
                })

                print(f"[OK] {gas_mixture} {chamber} {V}V -> mu = {v_mu1[0]:.1f} | S={signal_file.name} | BKG={bkg_file.name}")

            fout.cd()
            t.Write()
            fout.Close()

            print(f"[DONE] created {outname}")

    # --------------------------------------------------
    # RIEPILOGO FINALE UNICO
    # --------------------------------------------------
    global fit_summary_df
    fit_summary_df = pd.DataFrame(fit_summary_rows)

    if fit_summary_df.empty:
        print("\n===== RIEPILOGO FIT (VUOTO) =====")
        return fit_summary_df

    fit_summary_df = fit_summary_df.sort_values(
        ["miscela", "tensione_V", "camera"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    out_summary_dir = Path("output")
    out_summary_dir.mkdir(parents=True, exist_ok=True)
    out_summary_csv = out_summary_dir / "fit_summary_all_mixtures.csv"
    fit_summary_df.to_csv(out_summary_csv, index=False)

    print("\n===== RIEPILOGO FIT UNICO =====")
    print(fit_summary_df[["miscela", "camera", "tensione_V", "mu", "sigma", "chi2_ndf", "modello"]].to_string(index=False))
    print(f"\nCSV riepilogo salvato in: {out_summary_csv}")

    return fit_summary_df

In [21]:
main()


############################################
Processing mixture: Ar_CO2_Iso_93_5_2
############################################

Processing chamber: MM2DNAP1
[PNG] fit_results -> output/Ar_CO2_Iso_93_5_2/fit/fit_results
[PNG] bkg_histo  -> output/Ar_CO2_Iso_93_5_2/fit/bkg_histo
[PNG] tot_histo  -> output/Ar_CO2_Iso_93_5_2/fit/tot_histo
[BASE] Ar_CO2_Iso_93_5_2 MM2DNAP1 490V mu=452.0 (TSpectrum)
[QUALITY] GOOD {'snr': 1.8450704225352113, 'sigma': 178.03184504412002}
[XMIN] valley → 407.0
[XMIN_SIG] threshold=32.3 -> xmin>=401.0
[DEBUG] mu=452.0 sigma0=178.0 sigma_escape0=142.4 xmin=401.0 xmax=1342.2
[WEAK] snr_mu=1.85 weak=False
[COMPARE] 1g red=1.42 AIC=1283.3  2g red=0.96 AIC=874.8 amp_escape=141.4 mu_escape=316.4 sigma_escape=233.5 vis_escape=0.22 bkg@main/A0=0.68 force_escape=False
[MODEL] 1-gauss chi2/ndf=1.42
[OK] Ar_CO2_Iso_93_5_2 MM2DNAP1 490V -> mu = 725.6 | S=Ar_Co2_Iso_5_2_Fe55debole_MM2DNAP1_5APV_connesso_VerdeY1_490V_300v_grigio.txt | BKG=Ar_Co2_Iso_5_2_BKG_MM2DNAP1_5APV_c

,miscela,camera,tensione_V,mu,sigma,chi2,ndf,chi2_ndf,modello,signal_file,bkg_file
0,Ar_CO2_Iso_88_10_2,MM2DNAP1,490,1336.522461,5.000000,185.794556,450,0.412877,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
1,Ar_CO2_Iso_88_10_2,MM2DNAP1,500,838.074585,5.000000,154.284348,234,0.659335,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
2,Ar_CO2_Iso_88_10_2,MM2DNAP1,510,481.430847,26.267429,78.203743,64,1.221933,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
3,Ar_CO2_Iso_88_10_2,MM2DNAP1,520,1034.102417,15.936413,802.540894,290,2.767382,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
4,Ar_CO2_Iso_88_10_2,MM2DNAP1,530,644.711792,106.653030,1250.188232,749,1.669143,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
5,Ar_CO2_Iso_88_10_2,MM2DNAP1,540,887.415405,100.309593,1843.613403,901,2.046186,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
6,Ar_CO2_Iso_88_10_2,MM2DNAP1,550,1132.064453,131.600037,1311.290894,1166,1.124606,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
7,Ar_CO2_Iso_88_10_2,MM2DNAP1,560,1469.401855,164.837830,1643.986938,1510,1.088733,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
8,Ar_CO2_Iso_88_10_2,MM2DNAP1,570,2081.932129,273.532043,4352.411621,2634,1.652396,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...
9,Ar_CO2_Iso_88_10_2,MM2DNAP1,580,2777.277344,416.008026,10445.118164,3673,2.843757,1gauss,Ar_Co2_Iso_10_2_Fe55_MM2DNAP1_5APV_connesso_Ve...,Ar_Co2_Iso_10_2_BKG_MM2DNAP1_5APV_connesso_Ver...


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/fit_results/fit_MM2DNAP1_490V.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/fit_results/residuals_MM2DNAP1_490V.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/bkg_histo/bkg_MM2DNAP1_490V.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/tot_histo/tot_MM2DNAP1_490V.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/fit_results/fit_MM2DNAP1_500V.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/fit/fit_results/residuals_MM2DNAP1_500V.png has been created
Info in <TCanvas::Print>: png file output/

In [66]:
fcheck = ROOT.TFile("output_Ar_CO2_Iso_88_10_2_MM2DNAP1.root")
keys = [k.GetName() for k in fcheck.GetListOfKeys() if "h_residuals_" in k.GetName()]
print("N residual hist:", len(keys))
for name in sorted(keys)[:3]:
    htmp = fcheck.Get(name)
    print(name, "nbins=", htmp.GetNbinsX(), "xmin=", htmp.GetXaxis().GetXmin(), "xmax=", htmp.GetXaxis().GetXmax())
fcheck.Close()

N residual hist: 12
h_residuals_h_MM2DNAP1_490V nbins= 880 xmin= 477.5 xmax= 1357.5
h_residuals_h_MM2DNAP1_500V nbins= 338 xmin= 535.5 xmax= 873.5
h_residuals_h_MM2DNAP1_510V nbins= 73 xmin= 535.5 xmax= 608.5
